# CubeSat Telemetry Anomaly Detection: SOTA Optimization & Capacity Pareto Sweep
### Google Colab Ready — In-Domain, Cross-Mission, and Out-of-Domain Generalization

This notebook implements the complete research pipeline to maximize detection accuracy (**Raw-F1** and **Affiliation-F1**) across aerospace and industrial benchmarks while systematically finding the **accuracy vs. memory footprint Pareto frontier** ($421 \to 8,192$ parameters).

**Key Contributions:**
1. **Validation-Calibrated Thresholding:** Eliminates unsupervised heuristic bias via out-of-sample calibration ($\tau^* = \arg\max_\tau F1_{\text{val}}(\tau)$).
2. **Composite Anomaly Scoring:** Fuses reconstruction residual with first-difference rate-of-change residual ($S_t = \alpha S_{\text{recon}} + \beta S_{\Delta\text{recon}}$).
3. **Multi-Scale Edge Architecture:** Implements `MultiScaleTinyConvAE` with parallel multi-receptive fields ($k=3, 7, 15$).
4. **Anomaly-Aware Distillation:** Distills both reconstruction and anomaly pairwise ranking from a Teacher Ensemble.
5. **Empirical Capacity Sweep:** Maps the exact memory knee for onboard CubeSat microcontrollers (ARM Cortex-M4 / STM32F4).

## Phase 0: Environment Setup & Google Drive Mounting

In [ ]:
# Mount Google Drive (if running in Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    import os
    COLAB_ROOT = '/content/drive/MyDrive/cubesat_project'
    if os.path.exists(COLAB_ROOT):
        os.chdir(COLAB_ROOT)
        print(f'[Mounted Drive] Working directory set to: {os.getcwd()}')
except ImportError:
    print('[Local Execution] Running on local workspace.')

import torch
print(f'PyTorch Version: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Active Device: {DEVICE}')

## Phase 1: Multi-Scale Edge Architecture & Composite Scoring Engine

In [ ]:
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import precision_score, recall_score, f1_score

class MultiScaleConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        ch_div = max(1, out_channels // 3)
        rem = out_channels - 2 * ch_div
        self.conv3 = nn.Conv1d(in_channels, ch_div, kernel_size=3, padding=1)
        self.conv7 = nn.Conv1d(in_channels, ch_div, kernel_size=7, padding=3)
        self.conv15 = nn.Conv1d(in_channels, rem, kernel_size=15, padding=7)
        self.act = nn.ReLU()

    def forward(self, x):
        return self.act(torch.cat([self.conv3(x), self.conv7(x), self.conv15(x)], dim=1))

class ScalableMultiScaleStudent(nn.Module):
    def __init__(self, n_features=1, hidden_dim=6, latent_dim=3):
        super().__init__()
        self.enc1 = MultiScaleConvBlock(n_features, hidden_dim)
        self.enc2 = nn.Conv1d(hidden_dim, latent_dim, kernel_size=5, padding=2)
        self.dec1 = nn.Conv1d(latent_dim, hidden_dim, kernel_size=5, padding=2)
        self.dec2 = nn.Conv1d(hidden_dim, n_features, kernel_size=5, padding=2)
        self.relu = nn.ReLU()
        self.tanh = nn.Tanh()

    def forward(self, x):
        x_t = x.transpose(1, 2)
        h1 = self.enc1(x_t)
        z = self.relu(self.enc2(h1))
        h2 = self.relu(self.dec1(z))
        return self.tanh(self.dec2(h2)).transpose(1, 2)

print('[Engine Initialized] Multi-Scale Architecture ready.')

## Phase 2: Execute Validation-Calibrated Training & Capacity Pareto Sweep

In [ ]:
import os, sys
# Import standalone v3 runner
import generalization_v3.train_and_evaluate_v3 as v3

project_root = os.getcwd()
print(f'Starting full optimization pipeline from root: {project_root}')
results_df = v3.run_full_v3_optimization(project_root)
results_df

## Phase 3: Display Capacity vs. Accuracy Pareto Curve

In [ ]:
from IPython.display import Image, display
fig_path = os.path.join(project_root, 'generalization_v3', 'results', 'figures', 'capacity_vs_accuracy_pareto_curve.png')
if os.path.exists(fig_path):
    display(Image(fig_path))
else:
    print('Pareto curve figure not generated yet.')